# S&P 500 inclusion announcements — exploratory source-only archive

Original analytical notebook preserved as a research archive, with previously stored outputs removed. **Not rerun end-to-end for this publication.** Some cells repeat/replace earlier functions, ticker histories and announcement timestamps require independent checks, and availability-based exclusions can bias results. Do not interpret event-window returns as executable returns. For research scope and limitations see [the methodological note](../index-inclusion-event-study.md).


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt

# =========================
# 1. USER INPUTS
# =========================
stock_ticker = "PLTR"
market_ticker = "^GSPC"   # S&P 500
event_date = pd.Timestamp("2024-09-06")

# Tutorial defaults
estimation_start = -356
estimation_end = -11
event_window_start = -10
event_window_end = 10
main_car_start = -1
main_car_end = 1

# Pull a slightly wider range to be safe
download_start = "2023-01-01"
download_end = "2025-01-01"

# =========================
# 2. DOWNLOAD DATA
# =========================
prices = yf.download(
    [stock_ticker, market_ticker],
    start=download_start,
    end=download_end,
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices.dropna().copy()
prices.columns = ["stock_close", "market_close"]

# =========================
# 3. COMPUTE DAILY RETURNS
# =========================
prices["R_stock"] = prices["stock_close"].pct_change()
prices["R_market"] = prices["market_close"].pct_change()
prices = prices.dropna().copy()

# =========================
# 4. CREATE EVENT DAY INDEX
# =========================
# Find the row position of the event trading day
if event_date not in prices.index:
    raise ValueError(
        f"Event date {event_date.date()} is not in the trading-day index. "
        "Check whether the event date was a non-trading day and shift accordingly."
    )

event_loc = prices.index.get_loc(event_date)
prices["Event_Day"] = np.arange(len(prices)) - event_loc

# =========================
# 5. DEFINE WINDOWS
# =========================
estimation_mask = (prices["Event_Day"] >= estimation_start) & (prices["Event_Day"] <= estimation_end)
event_mask = (prices["Event_Day"] >= event_window_start) & (prices["Event_Day"] <= event_window_end)
main_car_mask = (prices["Event_Day"] >= main_car_start) & (prices["Event_Day"] <= main_car_end)

estimation_data = prices.loc[estimation_mask].copy()
event_data = prices.loc[event_mask].copy()

if len(estimation_data) < 100:
    raise ValueError("Too few observations in the estimation window. Expand the download range.")

# =========================
# 6. ESTIMATE MARKET MODEL
# =========================
X_est = sm.add_constant(estimation_data["R_market"])
y_est = estimation_data["R_stock"]

market_model = sm.OLS(y_est, X_est).fit()

alpha = market_model.params["const"]
beta = market_model.params["R_market"]
sigma_epsilon = market_model.resid.std(ddof=1)

print("=== Market Model Results ===")
print(f"Alpha       = {alpha:.6f}")
print(f"Beta        = {beta:.6f}")
print(f"R-squared   = {market_model.rsquared:.4f}")
print(f"Sigma_eps   = {sigma_epsilon:.6f}")
print()
print(market_model.summary())

# =========================
# 7. CALCULATE ABNORMAL RETURNS
# =========================
event_data["Expected_Return"] = alpha + beta * event_data["R_market"]
event_data["AR"] = event_data["R_stock"] - event_data["Expected_Return"]
event_data["CAR"] = event_data["AR"].cumsum()

# =========================
# 8. COMPUTE CARs
# =========================
car_main = event_data.loc[
    (event_data["Event_Day"] >= main_car_start) & (event_data["Event_Day"] <= main_car_end),
    "AR"
].sum()

car_wide = event_data["AR"].sum()

# t-statistic for CAR over a window of length N:
# t_CAR = CAR / (sigma_epsilon * sqrt(N))
N_main = main_car_end - main_car_start + 1
N_wide = event_window_end - event_window_start + 1

t_main = car_main / (sigma_epsilon * np.sqrt(N_main))
t_wide = car_wide / (sigma_epsilon * np.sqrt(N_wide))

print("\n=== Event Study Results ===")
print(f"CAR({main_car_start}, {main_car_end})  = {car_main:.6f}")
print(f"t-stat for CAR({main_car_start}, {main_car_end}) = {t_main:.4f}")
print(f"CAR({event_window_start}, {event_window_end}) = {car_wide:.6f}")
print(f"t-stat for CAR({event_window_start}, {event_window_end}) = {t_wide:.4f}")

# Optional p-value approximation from standard normal
from scipy.stats import norm
p_main = 2 * (1 - norm.cdf(abs(t_main)))
p_wide = 2 * (1 - norm.cdf(abs(t_wide)))

print(f"Approx. p-value main window = {p_main:.4f}")
print(f"Approx. p-value wide window = {p_wide:.4f}")

# =========================
# 9. DISPLAY EVENT TABLE
# =========================
display_cols = ["R_stock", "R_market", "Expected_Return", "AR", "CAR"]
print("\n=== Event Window Table ===")
print(event_data[["Event_Day"] + display_cols].round(6))

# =========================
# 10. PLOT CAR
# =========================
plt.figure(figsize=(10, 6))
plt.plot(event_data["Event_Day"], event_data["CAR"] * 100, marker="o")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.xlabel("Trading Days Relative to Event")
plt.ylabel("Cumulative Abnormal Return (%)")
plt.title(f"Event Study CAR: {stock_ticker} around {event_date.date()}")
plt.grid(True, alpha=0.3)
plt.show()

# =========================
# 11. OPTIONAL: DAILY AR BAR CHART
# =========================
plt.figure(figsize=(10, 6))
plt.bar(event_data["Event_Day"], event_data["AR"] * 100)
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.xlabel("Trading Days Relative to Event")
plt.ylabel("Abnormal Return (%)")
plt.title(f"Daily Abnormal Returns: {stock_ticker}")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

# =========================
# 1. SETTINGS
# =========================
stock_ticker = "PLTR"
benchmark_ticker = "^IXIC"   # Nasdaq Composite; use ^GSPC if you prefer S&P 500

start_date = "2023-01-01"
end_date   = "2026-01-31"

# Important Palantir events to annotate
events = {
    "2024-03-04": "Ukraine demining\npartnership",
    "2024-05-30": "DoD CDAO /\nCJADC2 selection",
    "2024-08-08": "Microsoft + Palantir\nclassified AI",
    "2024-09-06": "S&P 500\ninclusion announced",
    "2024-09-20": "Maven military\nservices expansion",
    "2024-11-04": "Q3 earnings +\nraised guidance",
    "2024-12-06": "Booz Allen +\nAnduril defense AI",
    "2024-12-18": "Army Vantage\n$618.9M expansion"
}

# =========================
# 2. DOWNLOAD DATA
# =========================
raw = yf.download(
    [stock_ticker, benchmark_ticker],
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

# yfinance sometimes returns MultiIndex columns
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].copy()
    prices.columns = [stock_ticker]

prices = prices.dropna().copy()
prices.columns = ["PLTR", "Benchmark"]

# =========================
# 3. CORE SERIES
# =========================
df = prices.copy()
df["PLTR_ret"] = df["PLTR"].pct_change()
df["Benchmark_ret"] = df["Benchmark"].pct_change()

# Indexed performance (start = 100)
df["PLTR_index"] = 100 * df["PLTR"] / df["PLTR"].iloc[0]
df["Benchmark_index"] = 100 * df["Benchmark"] / df["Benchmark"].iloc[0]

# Relative strength: PLTR divided by benchmark, rebased to 100
df["Relative_Strength"] = 100 * (df["PLTR"] / df["Benchmark"]) / (df["PLTR"].iloc[0] / df["Benchmark"].iloc[0])

# Rolling metrics
window = 20
df["PLTR_20d_return"] = df["PLTR"].pct_change(window)
df["Benchmark_20d_return"] = df["Benchmark"].pct_change(window)
df["PLTR_20d_vol"] = df["PLTR_ret"].rolling(window).std() * np.sqrt(252)
df["Benchmark_20d_vol"] = df["Benchmark_ret"].rolling(window).std() * np.sqrt(252)

# Rolling beta
rolling_cov = df["PLTR_ret"].rolling(window).cov(df["Benchmark_ret"])
rolling_var = df["Benchmark_ret"].rolling(window).var()
df["Rolling_Beta_20d"] = rolling_cov / rolling_var

# Rolling alpha from simple market model on a moving window
alphas = []
for i in range(len(df)):
    if i < window:
        alphas.append(np.nan)
    else:
        sample = df.iloc[i-window:i].dropna(subset=["PLTR_ret", "Benchmark_ret"])
        X = sm.add_constant(sample["Benchmark_ret"])
        y = sample["PLTR_ret"]
        model = sm.OLS(y, X).fit()
        alphas.append(model.params["const"])
df["Rolling_Alpha_20d"] = alphas

# Linear trend on indexed PLTR price
x = np.arange(len(df))
coef = np.polyfit(x, df["PLTR_index"], 1)
df["PLTR_linear_trend"] = coef[0] * x + coef[1]

# Optional moving averages
df["MA_20"] = df["PLTR"].rolling(20).mean()
df["MA_50"] = df["PLTR"].rolling(50).mean()

# =========================
# 4. HELPER: SNAP EVENT DATES TO TRADING DAYS
# =========================
def nearest_trading_day(index, date_str):
    target = pd.Timestamp(date_str)
    if target in index:
        return target
    # nearest available trading day
    pos = index.get_indexer([target], method="nearest")[0]
    return index[pos]

event_points = []
for d, label in events.items():
    td = nearest_trading_day(df.index, d)
    event_points.append((td, label))

# =========================
# 5. SUMMARY STATS
# =========================
def summarize_trend(data):
    total_return = data["PLTR"].iloc[-1] / data["PLTR"].iloc[0] - 1
    benchmark_return = data["Benchmark"].iloc[-1] / data["Benchmark"].iloc[0] - 1
    excess_return = total_return - benchmark_return

    mean_ret = data["PLTR_ret"].mean() * 252
    vol = data["PLTR_ret"].std() * np.sqrt(252)
    beta_full = data["PLTR_ret"].cov(data["Benchmark_ret"]) / data["Benchmark_ret"].var()

    print("===== TREND SUMMARY =====")
    print(f"Sample: {data.index.min().date()} to {data.index.max().date()}")
    print(f"PLTR total return:       {total_return:.2%}")
    print(f"Benchmark total return:  {benchmark_return:.2%}")
    print(f"PLTR excess return:      {excess_return:.2%}")
    print(f"Annualized mean return:  {mean_ret:.2%}")
    print(f"Annualized volatility:   {vol:.2%}")
    print(f"Full-sample beta:        {beta_full:.3f}")

    latest_rs = data['Relative_Strength'].iloc[-1]
    latest_beta = data["Rolling_Beta_20d"].dropna().iloc[-1] if data["Rolling_Beta_20d"].notna().any() else np.nan
    latest_vol = data["PLTR_20d_vol"].dropna().iloc[-1] if data["PLTR_20d_vol"].notna().any() else np.nan

    print(f"Latest relative strength index: {latest_rs:.2f}")
    print(f"Latest rolling beta (20d):      {latest_beta:.3f}")
    print(f"Latest rolling vol (20d ann.):  {latest_vol:.2%}")

summarize_trend(df.dropna())

# =========================
# 6. MAIN CHART: INDEXED PERFORMANCE + EVENTS
# =========================
plt.figure(figsize=(15, 8))
plt.plot(df.index, df["PLTR_index"], label="PLTR (Indexed to 100)", linewidth=2)
plt.plot(df.index, df["Benchmark_index"], label=f"{benchmark_ticker} (Indexed to 100)", linewidth=1.8)
plt.plot(df.index, df["PLTR_linear_trend"], linestyle="--", label="PLTR linear trend")

for i, (d, label) in enumerate(event_points):
    y = df.loc[d, "PLTR_index"]
    plt.axvline(d, linestyle="--", alpha=0.45)
    offset = 10 if i % 2 == 0 else -15
    plt.annotate(
        label,
        xy=(d, y),
        xytext=(10, offset),
        textcoords="offset points",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", alpha=0.15),
        arrowprops=dict(arrowstyle="->", lw=0.8)
    )

plt.title("Palantir: Full Indexed Price Path with Strategic Event Labels")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 7. PRICE + MOVING AVERAGES
# =========================
plt.figure(figsize=(15, 7))
plt.plot(df.index, df["PLTR"], label="PLTR Price", linewidth=2)
plt.plot(df.index, df["MA_20"], label="20-day MA", linewidth=1.5)
plt.plot(df.index, df["MA_50"], label="50-day MA", linewidth=1.5)

for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.3)

plt.title("Palantir Price with 20-Day and 50-Day Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 8. RELATIVE STRENGTH
# =========================
plt.figure(figsize=(15, 6))
plt.plot(df.index, df["Relative_Strength"], linewidth=2)
for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.25)
plt.axhline(100, linestyle="--")
plt.title("Palantir Relative Strength vs Benchmark (Rebased to 100)")
plt.xlabel("Date")
plt.ylabel("Relative Strength Index")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 9. ROLLING BETA
# =========================
plt.figure(figsize=(15, 6))
plt.plot(df.index, df["Rolling_Beta_20d"], linewidth=2)
for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.25)
plt.axhline(1.0, linestyle="--")
plt.title("Palantir 20-Day Rolling Beta vs Benchmark")
plt.xlabel("Date")
plt.ylabel("Rolling Beta")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 10. ROLLING VOLATILITY
# =========================
plt.figure(figsize=(15, 6))
plt.plot(df.index, df["PLTR_20d_vol"], label="PLTR 20d annualized vol", linewidth=2)
plt.plot(df.index, df["Benchmark_20d_vol"], label="Benchmark 20d annualized vol", linewidth=1.8)
for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.25)
plt.title("Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 11. 20-DAY FORWARD RETURN AFTER EACH EVENT
# =========================
print("\n===== EVENT-BY-EVENT FORWARD RETURN CHECK =====")
rows = []
for d, label in event_points:
    if d not in df.index:
        continue
    start_px = df.loc[d, "PLTR"]
    # nearest 20 trading days later
    idx_pos = df.index.get_loc(d)
    end_pos = min(idx_pos + 20, len(df) - 1)
    end_d = df.index[end_pos]
    end_px = df.iloc[end_pos]["PLTR"]
    fwd_ret = end_px / start_px - 1

    rows.append({
        "Event_Date": d.date(),
        "Label": label.replace("\n", " "),
        "PLTR_Price_On_Event": round(start_px, 2),
        "20D_Later": end_d.date(),
        "20D_Forward_Return": fwd_ret
    })

event_forward = pd.DataFrame(rows)
print(event_forward.to_string(index=False))

# =========================
# 12. OPTIONAL: SIMPLE TREND REGIMES
# =========================
df["Above_MA20"] = (df["PLTR"] > df["MA_20"]).astype(int)
df["Above_MA50"] = (df["PLTR"] > df["MA_50"]).astype(int)

print("\n===== LATEST TREND REGIME =====")
latest = df.dropna().iloc[-1]
print(f"Latest date:            {df.dropna().index[-1].date()}")
print(f"Price above 20d MA:     {bool(latest['Above_MA20'])}")
print(f"Price above 50d MA:     {bool(latest['Above_MA50'])}")
print(f"20d rolling beta:       {latest['Rolling_Beta_20d']:.3f}")
print(f"20d rolling vol:        {latest['PLTR_20d_vol']:.2%}")
print(f"Relative strength idx:  {latest['Relative_Strength']:.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# event_data already contains: Event_Day, AR, CAR
event_data = event_data.copy().sort_values("Event_Day")

# Number of accumulated days from the start of the event window
event_data["N_days"] = np.arange(1, len(event_data) + 1)

# Standard error of CAR at each horizon
event_data["CAR_SE"] = sigma_epsilon * np.sqrt(event_data["N_days"])

# 95% confidence bands around zero under null of no abnormal performance
event_data["Upper_95"] = 1.96 * event_data["CAR_SE"]
event_data["Lower_95"] = -1.96 * event_data["CAR_SE"]

# Optional: confidence bands around the estimated CAR itself
event_data["CAR_upper_band"] = event_data["CAR"] + 1.96 * event_data["CAR_SE"]
event_data["CAR_lower_band"] = event_data["CAR"] - 1.96 * event_data["CAR_SE"]

plt.figure(figsize=(11, 6))
plt.plot(event_data["Event_Day"], event_data["CAR"], label="CAR", marker="o")
plt.plot(event_data["Event_Day"], event_data["Upper_95"], linestyle="--", label="Upper 95% bound")
plt.plot(event_data["Event_Day"], event_data["Lower_95"], linestyle="--", label="Lower 95% bound")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.xlabel("Trading Days Relative to Event")
plt.ylabel("Cumulative Abnormal Return")
plt.title("CAR with 95% Confidence Bounds")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(event_data["Event_Day"], event_data["CAR"], label="CAR", marker="o")
plt.fill_between(
    event_data["Event_Day"],
    event_data["Lower_95"],
    event_data["Upper_95"],
    alpha=0.2,
    label="95% confidence region under H0: CAR = 0"
)
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.xlabel("Trading Days Relative to Event")
plt.ylabel("Cumulative Abnormal Return")
plt.title("CAR with 95% Confidence Region")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

As of **17 March 2026**, the **20 most recent S&P 500 additions by effective date** are these:

1. **Vertiv Holdings (VRT)** — **23 Mar 2026** ([News Release Archive][1])
2. **Lumentum Holdings (LITE)** — **23 Mar 2026** ([News Release Archive][1])
3. **Coherent (COHR)** — **23 Mar 2026** ([News Release Archive][1])
4. **EchoStar (SATS)** — **23 Mar 2026** ([News Release Archive][1])
5. **Ciena (CIEN)** — **9 Feb 2026** ([News Release Archive][2])
6. **CRH (CRH)** — **22 Dec 2025** ([News Release Archive][3])
7. **Carvana (CVNA)** — **22 Dec 2025** ([News Release Archive][3])
8. **Comfort Systems USA (FIX)** — **22 Dec 2025** ([News Release Archive][3])
9. **Ares Management (ARES)** — **11 Dec 2025** ([News Release Archive][4])
10. **Sandisk (SNDK)** — **28 Nov 2025** ([News Release Archive][5])
11. **Block (XYZ)** — **23 Jul 2025** ([News Release Archive][6])
12. **Coinbase Global (COIN)** — **19 May 2025** ([News Release Archive][7])
13. **Apollo Global Management (APO)** — **23 Dec 2024** ([News Release Archive][8])
14. **Workday (WDAY)** — **23 Dec 2024** ([News Release Archive][8])
15. **Lennox International (LII)** — **23 Dec 2024** ([News Release Archive][9])
16. **Amentum Holdings (AMTM)** — **30 Sep 2024** ([News Release Archive][10])
17. **Palantir Technologies (PLTR)** — **23 Sep 2024** ([News Release Archive][11])
18. **Dell Technologies (DELL)** — **23 Sep 2024** ([News Release Archive][11])
19. **Erie Indemnity (ERIE)** — **23 Sep 2024** ([News Release Archive][11])
20. **GE Vernova (GEV)** — **2 Apr 2024** ([News Release Archive][12])

[1]: https://press.spglobal.com/2026-03-06-Vertiv-Holdings%2C-Lumentum-Holdings%2C-Coherent%2C-and-EchoStar-Set-to-Join-S-P-500-Others-to-Join-S-P-100%2C-S-P-MidCap-400%2C-and-S-P-SmallCap-600 "Vertiv Holdings, Lumentum Holdings, Coherent, and EchoStar Set to Join S&P 500; Others to Join S&P 100, S&P MidCap 400, and S&P SmallCap 600 - Mar 6, 2026"
[2]: https://press.spglobal.com/2026-02-04-Ciena-Set-to-Join-S-P-500-Arrowhead-Pharmaceuticals-to-Join-S-P-MidCap-400-ADT-and-OneSpaWorld-Holdings-to-Join-S-P-SmallCap-600 "Ciena Set to Join S&P 500; Arrowhead Pharmaceuticals to Join S&P MidCap 400; ADT and OneSpaWorld Holdings to Join S&P SmallCap 600 - Feb 4, 2026"
[3]: https://press.spglobal.com/2025-12-05-CRH%2C-Carvana-and-Comfort-Systems-USA-Set-to-Join-S-P-500-Others-to-Join-S-P-MidCap-400-and-S-P-SmallCap-600 "CRH, Carvana and Comfort Systems USA Set to Join S&P 500; Others to Join S&P MidCap 400 and S&P SmallCap 600 - Dec 5, 2025"
[4]: https://press.spglobal.com/2025-12-08-Ares-Management-Set-to-Join-S-P-500-Sezzle-and-Vital-Farms-to-Join-S-P-SmallCap-600 "Ares Management Set to Join S&P 500; Sezzle and Vital Farms to Join S&P SmallCap 600 - Dec 8, 2025"
[5]: https://press.spglobal.com/2025-11-24-Sandisk-Set-to-Join-S-P-500-Upwork%2C-First-Interstate-BancSystem%2C-PTC-Therapeutics-to-Join-S-P-SmallCap-600 "Sandisk Set to Join S&P 500; Upwork, First Interstate BancSystem, PTC Therapeutics to Join S&P SmallCap 600 - Nov 24, 2025"
[6]: https://press.spglobal.com/2025-07-18-Block-Set-to-Join-S-P-500 "Block Set to Join S&P 500 - Jul 18, 2025"
[7]: https://press.spglobal.com/2025-05-12-Coinbase-Global-Set-to-Join-S-P-500 "Coinbase Global Set to Join S&P 500 - May 12, 2025"
[8]: https://press.spglobal.com/2024-12-06-Apollo-Global-Management-and-Workday-Set-to-Join-S-P-500-Others-to-Join-S-P-MidCap-400-and-S-P-SmallCap-600 "Apollo Global Management and Workday Set to Join S&P 500; Others to Join S&P MidCap 400 and S&P SmallCap 600 - Dec 6, 2024"
[9]: https://press.spglobal.com/2024-12-18-Lennox-International-Set-to-Join-S-P-500-and-BILL-Holdings-to-Join-S-P-MidCap-400 "Lennox International Set to Join S&P 500 and BILL Holdings to Join S&P MidCap 400 - Dec 18, 2024"
[10]: https://press.spglobal.com/2024-09-24-Amentum-Set-to-Join-S-P-500-Bath-Body-Works-to-Join-S-P-SmallCap-600 "Amentum Set to Join S&P 500; Bath & Body Works to Join S&P SmallCap 600 - Sep 24, 2024"
[11]: https://press.spglobal.com/2024-09-06-Palantir-Technologies%2C-Dell-Technologies%2C-and-Erie-Indemnity-Set-to-Join-S-P-500-Others-to-Join-S-P-MidCap-400-and-S-P-SmallCap-600 "Palantir Technologies, Dell Technologies, and Erie Indemnity Set to Join S&P 500; Others to Join S&P MidCap 400 and S&P SmallCap 600 - Sep 6, 2024"
[12]: https://press.spglobal.com/2024-03-27-GE-Vernova-and-Solventum-Set-to-Join-S-P-500-Dentsply-Sirona-to-Join-S-P-MidCap-400-Others-to-Join-S-P-SmallCap-600 "GE Vernova and Solventum Set to Join S&P 500; Dentsply Sirona to Join S&P MidCap 400; Others to Join S&P SmallCap 600 - Mar 27, 2024"

here are the **exact S&P Dow Jones Indices announcement dates** for those same 20 recent S&P 500 additions.

* **Vertiv Holdings (VRT)** — announced **6 Mar 2026**.
  **Lumentum Holdings (LITE)** — announced **6 Mar 2026**.
  **Coherent (COHR)** — announced **6 Mar 2026**.
  **EchoStar (SATS)** — announced **6 Mar 2026**. ([News Release Archive][1])

* **Ciena (CIEN)** — announced **4 Feb 2026**. ([News Release Archive][2])

* **CRH (CRH)** — announced **5 Dec 2025**.
  **Carvana (CVNA)** — announced **5 Dec 2025**.
  **Comfort Systems USA (FIX)** — announced **5 Dec 2025**. ([News Release Archive][3])

* **Ares Management (ARES)** — announced **8 Dec 2025**. ([News Release Archive][4])

* **Sandisk (SNDK)** — announced **24 Nov 2025**. ([News Release Archive][5])

* **Block (XYZ)** — announced **18 Jul 2025**. ([News Release Archive][6])

* **Coinbase Global (COIN)** — announced **12 May 2025**. ([News Release Archive][7])

* **Apollo Global Management (APO)** — announced **6 Dec 2024**.
  **Workday (WDAY)** — announced **6 Dec 2024**. ([News Release Archive][8])

* **Lennox International (LII)** — announced **18 Dec 2024**. This came in its own S&P release, separate from the Apollo/Workday release. ([News Release Archive][5])

* **Amentum Holdings (AMTM)** — announced **24 Sep 2024**. ([News Release Archive][8])

* **Palantir Technologies (PLTR)** — announced **6 Sep 2024**.
  **Dell Technologies (DELL)** — announced **6 Sep 2024**.
  **Erie Indemnity (ERIE)** — announced **6 Sep 2024**. ([News Release Archive][1])*GE Vernova (GEV)** — announced **27 Mar 2024**.
  **Solventum (SOLV)** — announced **27 Mar 2024**.  important practical note for your event study: the **announcement date** and the **first full trading day of reaction** are not always the same. For example, Palantir was announced on **Friday, 6 Sep 2024**, but a large part of the market reaction showed up on **Monday, 9 Sep 2024**. That distinction matters a great deal when you choose (t=0). ([News Release Archive][1])I can turn this into a clean Python list or DataFrame so you can test all 20 inclusion announcements in one batch.

[1]: https://press.spglobal.com/2026-03-06-Vertiv-Holdings%2C-Lumentum-Holdings%2C-Coherent%2C-and-EchoStar-Set-to-Join-S-P-500-Others-to-Join-S-P-100%2C-S-P-MidCap-400%2C-and-S-P-SmallCap-600?utm_source=chatgpt.com "Vertiv Holdings, Lumentum Holdings, Coherent, and EchoStar ..."
[2]: https://press.spglobal.com/2026-02-04-Ciena-Set-to-Join-S-P-500-Arrowhead-Pharmaceuticals-to-Join-S-P-MidCap-400-ADT-and-OneSpaWorld-Holdings-to-Join-S-P-SmallCap-600?utm_source=chatgpt.com "ADT and OneSpaWorld Holdings to Join S&P SmallCap 600"
[3]: https://press.spglobal.com/2025-12-05-CRH%2C-Carvana-and-Comfort-Systems-USA-Set-to-Join-S-P-500-Others-to-Join-S-P-MidCap-400-and-S-P-SmallCap-600?utm_source=chatgpt.com "CRH, Carvana and Comfort Systems USA Set to Join S&P 500"
[4]: https://press.spglobal.com/2025-12-08-Ares-Management-Set-to-Join-S-P-500-Sezzle-and-Vital-Farms-to-Join-S-P-SmallCap-600?utm_source=chatgpt.com "Ares Management Set to Join S&P 500; Sezzle and Vital ..."
[5]: https://press.spglobal.com/index.php?l=100&s=2429&utm_source=chatgpt.com "News Release Archive"
[6]: https://press.spglobal.com/2025-07-18-Block-Set-to-Join-S-P-500?utm_source=chatgpt.com "Block Set to Join S&P 500 - Jul 18, 2025 - Press Releases"
[7]: https://press.spglobal.com/2025-05-12-Coinbase-Global-Set-to-Join-S-P-500?utm_source=chatgpt.com "Coinbase Global Set to Join S&P 500 - May 12, 2025"
[8]: https://press.spglobal.com/index.php?l=100&o=200&s=2429&utm_source=chatgpt.com "News Release Archive"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import t

# =========================================================
# 1. INPUT: S&P 500 inclusion announcements
#    announcement_date = S&P DJI press release date
# =========================================================
events = [
    {"ticker": "PLTR", "name": "Palantir", "announcement_date": "2024-09-06"},
    {"ticker": "DELL", "name": "Dell", "announcement_date": "2024-09-06"},
    {"ticker": "ERIE", "name": "Erie Indemnity", "announcement_date": "2024-09-06"},
    {"ticker": "AMTM", "name": "Amentum", "announcement_date": "2024-09-24"},
    {"ticker": "LII",  "name": "Lennox", "announcement_date": "2024-12-18"},
    {"ticker": "APO",  "name": "Apollo Global Management", "announcement_date": "2024-12-06"},
    {"ticker": "WDAY", "name": "Workday", "announcement_date": "2024-12-06"},
    {"ticker": "COIN", "name": "Coinbase", "announcement_date": "2025-05-12"},
    {"ticker": "XYZ",  "name": "Block", "announcement_date": "2025-07-18"},
    {"ticker": "SNDK", "name": "Sandisk", "announcement_date": "2025-11-24"},
    {"ticker": "ARES", "name": "Ares Management", "announcement_date": "2025-12-08"},
    {"ticker": "CRH",  "name": "CRH", "announcement_date": "2025-12-05"},
    {"ticker": "CVNA", "name": "Carvana", "announcement_date": "2025-12-05"},
    {"ticker": "FIX",  "name": "Comfort Systems USA", "announcement_date": "2025-12-05"},
    {"ticker": "CIEN", "name": "Ciena", "announcement_date": "2026-02-04"},
    {"ticker": "VRT",  "name": "Vertiv", "announcement_date": "2026-03-06"},
    {"ticker": "LITE", "name": "Lumentum", "announcement_date": "2026-03-06"},
    {"ticker": "COHR", "name": "Coherent", "announcement_date": "2026-03-06"},
    {"ticker": "SATS", "name": "EchoStar", "announcement_date": "2026-03-06"},
]

benchmark_ticker = "^GSPC"

# Windows
EST_START = -131
EST_END   = -11
EVT_START = -10
EVT_END   = 10

# buffer for download
DOWNLOAD_BUFFER_BEFORE = 220
DOWNLOAD_BUFFER_AFTER = 40

# =========================================================
# 2. HELPERS
# =========================================================
def nearest_trading_day(index, date_str):
    target = pd.Timestamp(date_str)
    if target in index:
        return target
    pos = index.get_indexer([target], method="nearest")[0]
    return index[pos]

def yahoo_download_one(ticker, start_date, end_date):
    raw = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False
    )

    if raw.empty:
        return None

    cols = raw.columns
    if isinstance(cols, pd.MultiIndex):
        # flatten if needed
        raw.columns = [c[0] if isinstance(c, tuple) else c for c in raw.columns]

    keep_cols = [c for c in ["Close", "Volume"] if c in raw.columns]
    out = raw[keep_cols].copy()
    out.columns = [f"{ticker}_close" if c == "Close" else f"{ticker}_volume" for c in out.columns]
    return out

def compute_event_panel_for_stock(ticker, name, announcement_date, benchmark_ticker):
    ann_date = pd.Timestamp(announcement_date)

    start_date = (ann_date - pd.Timedelta(days=DOWNLOAD_BUFFER_BEFORE)).strftime("2023-01-01")
    end_date   = (ann_date + pd.Timedelta(days=DOWNLOAD_BUFFER_AFTER)).strftime("2026-03-15")

    stock = yahoo_download_one(ticker, start_date, end_date)
    market = yahoo_download_one(benchmark_ticker, start_date, end_date)

    if stock is None or market is None:
        return None

    df = stock.join(market, how="inner").dropna().copy()

    stock_close_col = f"{ticker}_close"
    stock_vol_col = f"{ticker}_volume"
    mkt_close_col = f"{benchmark_ticker}_close"

    if stock_close_col not in df.columns or mkt_close_col not in df.columns:
        return None

    df["R_stock"] = df[stock_close_col].pct_change()
    df["R_market"] = df[mkt_close_col].pct_change()
    df["log_volume"] = np.log(df[stock_vol_col].replace(0, np.nan))
    df = df.dropna().copy()

    if len(df) < 170:
        return None

    event_trading_day = nearest_trading_day(df.index, announcement_date)
    event_loc = df.index.get_loc(event_trading_day)
    df["event_day"] = np.arange(len(df)) - event_loc

    est_mask = (df["event_day"] >= EST_START) & (df["event_day"] <= EST_END)
    evt_mask = (df["event_day"] >= EVT_START) & (df["event_day"] <= EVT_END)

    est = df.loc[est_mask].copy()
    evt = df.loc[evt_mask].copy()

    if len(est) < 100 or len(evt) != (EVT_END - EVT_START + 1):
        return None

    # -----------------------
    # Market model for returns
    # -----------------------
    X = sm.add_constant(est["R_market"])
    y = est["R_stock"]
    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params["R_market"]
    sigma_eps = model.resid.std(ddof=1)

    evt["expected_return"] = alpha + beta * evt["R_market"]
    evt["AR"] = evt["R_stock"] - evt["expected_return"]

    # -----------------------
    # Abnormal volume
    # -----------------------
    mean_log_vol_est = est["log_volume"].mean()
    sd_log_vol_est = est["log_volume"].std(ddof=1)

    evt["AVOL"] = evt["log_volume"] - mean_log_vol_est

    # standardised version can also be useful
    if sd_log_vol_est > 0:
        evt["SAVOL"] = evt["AVOL"] / sd_log_vol_est
    else:
        evt["SAVOL"] = np.nan

    evt["ticker"] = ticker
    evt["name"] = name
    evt["announcement_date"] = ann_date
    evt["event_trading_day"] = event_trading_day
    evt["alpha"] = alpha
    evt["beta"] = beta
    evt["sigma_eps"] = sigma_eps
    evt["sigma_logvol"] = sd_log_vol_est

    return evt.reset_index().rename(columns={"index": "date"})

# =========================================================
# 3. BUILD STACKED EVENT PANEL
# =========================================================
panels = []
failed = []

for e in events:
    out = compute_event_panel_for_stock(
        ticker=e["ticker"],
        name=e["name"],
        announcement_date=e["announcement_date"],
        benchmark_ticker=benchmark_ticker
    )
    if out is None:
        failed.append(e["ticker"])
    else:
        panels.append(out)

if not panels:
    raise ValueError("No event panels were successfully built.")

event_panel = pd.concat(panels, ignore_index=True)

print("Successfully processed tickers:", sorted(event_panel["ticker"].unique()))
print("Failed tickers:", failed)

# =========================================================
# 4. AGGREGATE BY EVENT DAY
# =========================================================
def aggregate_with_ci(df, value_col):
    g = df.groupby("event_day")[value_col].agg(["mean", "std", "count"]).reset_index()
    g.rename(columns={"mean": "avg", "std": "sd", "count": "n"}, inplace=True)
    g["se"] = g["sd"] / np.sqrt(g["n"])

    # 95% CI using t critical value
    g["t_crit"] = g["n"].apply(lambda n: t.ppf(0.975, df=max(n - 1, 1)))
    g["ci_low"] = g["avg"] - g["t_crit"] * g["se"]
    g["ci_high"] = g["avg"] + g["t_crit"] * g["se"]
    return g.sort_values("event_day").copy()

aar = aggregate_with_ci(event_panel, "AR")
aav = aggregate_with_ci(event_panel, "AVOL")
saav = aggregate_with_ci(event_panel, "SAVOL")

# CAAR
aar["CAAR"] = aar["avg"].cumsum()

# Approximate CAAR variance from cross-sectional daily AAR SEs
aar["CAAR_var"] = (aar["se"] ** 2).cumsum()
aar["CAAR_se"] = np.sqrt(aar["CAAR_var"])
aar["CAAR_ci_low"] = aar["CAAR"] - 1.96 * aar["CAAR_se"]
aar["CAAR_ci_high"] = aar["CAAR"] + 1.96 * aar["CAAR_se"]

# =========================================================
# 5. SUMMARY TABLES
# =========================================================
summary_rows = []
for ticker, sub in event_panel.groupby("ticker"):
    sub = sub.sort_values("event_day")
    car_3 = sub.loc[sub["event_day"].between(-1, 1), "AR"].sum()
    car_21 = sub.loc[sub["event_day"].between(-10, 10), "AR"].sum()
    avg_avol_3 = sub.loc[sub["event_day"].between(-1, 1), "AVOL"].mean()
    summary_rows.append({
        "ticker": ticker,
        "name": sub["name"].iloc[0],
        "announcement_date": sub["announcement_date"].iloc[0].date(),
        "event_trading_day": pd.Timestamp(sub["event_trading_day"].iloc[0]).date(),
        "CAR(-1,1)": car_3,
        "CAR(-10,10)": car_21,
        "Avg_AVOL(-1,1)": avg_avol_3
    })

summary_df = pd.DataFrame(summary_rows).sort_values("announcement_date")
print("\nEvent-level summary:")
print(summary_df.to_string(index=False))

# =========================================================
# 6. PLOTS
# =========================================================
plt.figure(figsize=(10, 6))
plt.plot(aar["event_day"], aar["avg"], marker="o", label="AAR")
plt.fill_between(aar["event_day"], aar["ci_low"], aar["ci_high"], alpha=0.2, label="95% CI")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.title("Average Abnormal Return (AAR) around S&P 500 Inclusion Announcements")
plt.xlabel("Trading Days Relative to Announcement")
plt.ylabel("AAR")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(aar["event_day"], aar["CAAR"], marker="o", label="CAAR")
plt.fill_between(aar["event_day"], aar["CAAR_ci_low"], aar["CAAR_ci_high"], alpha=0.2, label="95% CI")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.title("Cumulative Average Abnormal Return (CAAR)")
plt.xlabel("Trading Days Relative to Announcement")
plt.ylabel("CAAR")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(aav["event_day"], aav["avg"], marker="o", label="Average Abnormal Log-Volume")
plt.fill_between(aav["event_day"], aav["ci_low"], aav["ci_high"], alpha=0.2, label="95% CI")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.title("Average Abnormal Trading Volume around S&P 500 Inclusion Announcements")
plt.xlabel("Trading Days Relative to Announcement")
plt.ylabel("Average Abnormal Log-Volume")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(saav["event_day"], saav["avg"], marker="o", label="Standardised Abnormal Volume")
plt.fill_between(saav["event_day"], saav["ci_low"], saav["ci_high"], alpha=0.2, label="95% CI")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.title("Average Standardised Abnormal Volume")
plt.xlabel("Trading Days Relative to Announcement")
plt.ylabel("Standardised Abnormal Volume")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================================================
# 7. SMALL MULTIPLES: INDIVIDUAL EVENT CARS
# =========================================================
tickers = summary_df["ticker"].tolist()
n = len(tickers)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

for ax, ticker in zip(axes, tickers):
    sub = event_panel[event_panel["ticker"] == ticker].sort_values("event_day").copy()
    sub["CAR"] = sub["AR"].cumsum()

    sigma = sub["sigma_eps"].iloc[0]
    sub["N_days"] = np.arange(1, len(sub) + 1)
    sub["upper"] = 1.96 * sigma * np.sqrt(sub["N_days"])
    sub["lower"] = -1.96 * sigma * np.sqrt(sub["N_days"])

    ax.plot(sub["event_day"], sub["CAR"], marker="o", linewidth=1.5)
    ax.fill_between(sub["event_day"], sub["lower"], sub["upper"], alpha=0.15)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(f"{ticker}\n{str(sub['announcement_date'].iloc[0])[:10]}")
    ax.grid(alpha=0.3)

for j in range(len(tickers), len(axes)):
    axes[j].axis("off")

fig.suptitle("Individual Event CARs with 95% Null Reference Bands", y=1.02, fontsize=14)
fig.supxlabel("Trading Days Relative to Announcement")
fig.supylabel("CAR")
plt.tight_layout()
plt.savefig("confidence-intervals.png")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# ---------------------------------------------------------
# 1. Bootstrap helper for one event
# ---------------------------------------------------------
def bootstrap_car_envelope(estimation_df, event_df, n_boot=5000, random_state=42):
    """
    estimation_df must contain:
        R_stock, R_market
    event_df must contain:
        event_day, R_market, AR
    Returns:
        event_df with CAR, lower_95, upper_95
    """
    rng = np.random.default_rng(random_state)

    # Fit market model on estimation window
    X_est = sm.add_constant(estimation_df["R_market"])
    y_est = estimation_df["R_stock"]
    model = sm.OLS(y_est, X_est).fit()

    alpha = model.params["const"]
    beta = model.params["R_market"]
    residuals = model.resid.dropna().values

    # Actual event-window abnormal returns and CAR
    out = event_df.copy().sort_values("event_day")
    out["expected_return"] = alpha + beta * out["R_market"]
    out["AR"] = out["R_stock"] - out["expected_return"]
    out["CAR"] = out["AR"].cumsum()

    # Bootstrap null CAR paths:
    # simulate event-window abnormal returns by resampling estimation residuals
    T = len(out)
    boot_cars = np.zeros((n_boot, T))

    for b in range(n_boot):
        sampled_eps = rng.choice(residuals, size=T, replace=True)
        boot_cars[b, :] = np.cumsum(sampled_eps)

    out["lower_95"] = np.percentile(boot_cars, 2.5, axis=0)
    out["upper_95"] = np.percentile(boot_cars, 97.5, axis=0)

    return out, model


# ---------------------------------------------------------
# 2. Rebuild per-stock panels with bootstrap envelopes
# ---------------------------------------------------------
# Assumes you still have event_panel already built
# and original window settings EVT_START, EVT_END.

individual_results = {}

for ticker in sorted(event_panel["ticker"].unique()):
    sub = event_panel[event_panel["ticker"] == ticker].copy().sort_values("event_day")

    # Reconstruct estimation data from alpha/beta is not enough;
    # we need the original price panel again, so let us download it cleanly.
    meta = sub.iloc[0]
    ann_date = pd.Timestamp(meta["announcement_date"])
    name = meta["name"]

    start_date = (ann_date - pd.Timedelta(days=220)).strftime("%Y-%m-%d")
    end_date   = (ann_date + pd.Timedelta(days=40)).strftime("%Y-%m-%d")

    # Download again
    stock = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)
    market = yf.download("^GSPC", start=start_date, end=end_date, auto_adjust=True, progress=False)

    if stock.empty or market.empty:
        continue

    stock = stock[["Close"]].rename(columns={"Close": "stock_close"})
    if "Volume" in yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False).columns:
        pass  # not needed here

    market = market[["Close"]].rename(columns={"Close": "market_close"})

    df = stock.join(market, how="inner").dropna().copy()
    df["R_stock"] = df["stock_close"].pct_change()
    df["R_market"] = df["market_close"].pct_change()
    df = df.dropna().copy()

    if ann_date not in df.index:
        pos = df.index.get_indexer([ann_date], method="nearest")[0]
        event_trading_day = df.index[pos]
    else:
        event_trading_day = ann_date

    event_loc = df.index.get_loc(event_trading_day)
    df["event_day"] = np.arange(len(df)) - event_loc

    estimation_df = df[(df["event_day"] >= EST_START) & (df["event_day"] <= EST_END)].copy()
    event_df = df[(df["event_day"] >= EVT_START) & (df["event_day"] <= EVT_END)].copy()

    if len(estimation_df) < 80 or len(event_df) != (EVT_END - EVT_START + 1):
        continue

    boot_df, model = bootstrap_car_envelope(
        estimation_df=estimation_df,
        event_df=event_df[["event_day", "R_stock", "R_market"]].copy(),
        n_boot=5000,
        random_state=42
    )

    boot_df["ticker"] = ticker
    boot_df["name"] = name
    boot_df["announcement_date"] = ann_date

    individual_results[ticker] = boot_df


# ---------------------------------------------------------
# 3. Plot small multiples with bootstrap envelopes
# ---------------------------------------------------------
tickers = sorted(individual_results.keys())
n = len(tickers)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

for ax, ticker in zip(axes, tickers):
    sub = individual_results[ticker].copy().sort_values("event_day")
    ann_date = pd.Timestamp(sub["announcement_date"].iloc[0]).date()

    ax.plot(sub["event_day"], sub["CAR"], marker="o", linewidth=1.6, label="CAR")
    ax.fill_between(
        sub["event_day"],
        sub["lower_95"],
        sub["upper_95"],
        alpha=0.18,
        label="Bootstrap 95% null envelope"
    )
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(f"{ticker}\n{ann_date}")
    ax.grid(alpha=0.3)

for j in range(len(tickers), len(axes)):
    axes[j].axis("off")

fig.suptitle("Individual Event CARs with Bootstrap 95% Null Envelopes", y=1.02, fontsize=14)
fig.supxlabel("Trading Days Relative to Announcement")
fig.supylabel("CAR")
plt.tight_layout()
plt.savefig("confidence-intervals-1.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

# ---------------------------------------------------------
# 1. Build per-stock AR panels, not CAR bands
# ---------------------------------------------------------
individual_ar_results = {}

for ticker in sorted(event_panel["ticker"].unique()):
    sub = event_panel[event_panel["ticker"] == ticker].copy().sort_values("event_day")
    meta = sub.iloc[0]
    ann_date = pd.Timestamp(meta["announcement_date"])
    name = meta["name"]

    start_date = (ann_date - pd.Timedelta(days=220)).strftime("%Y-%m-%d")
    end_date   = (ann_date + pd.Timedelta(days=40)).strftime("%Y-%m-%d")

    stock = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)
    market = yf.download("^GSPC", start=start_date, end=end_date, auto_adjust=True, progress=False)

    if stock.empty or market.empty:
        continue

    stock = stock[["Close"]].rename(columns={"Close": "stock_close"})
    market = market[["Close"]].rename(columns={"Close": "market_close"})

    df = stock.join(market, how="inner").dropna().copy()
    df["R_stock"] = df["stock_close"].pct_change()
    df["R_market"] = df["market_close"].pct_change()
    df = df.dropna().copy()

    if ann_date not in df.index:
        pos = df.index.get_indexer([ann_date], method="nearest")[0]
        event_trading_day = df.index[pos]
    else:
        event_trading_day = ann_date

    event_loc = df.index.get_loc(event_trading_day)
    df["event_day"] = np.arange(len(df)) - event_loc

    estimation_df = df[(df["event_day"] >= EST_START) & (df["event_day"] <= EST_END)].copy()
    event_df = df[(df["event_day"] >= EVT_START) & (df["event_day"] <= EVT_END)].copy()

    if len(estimation_df) < 80 or len(event_df) != (EVT_END - EVT_START + 1):
        continue

    # Market model
    X_est = sm.add_constant(estimation_df["R_market"])
    y_est = estimation_df["R_stock"]
    model = sm.OLS(y_est, X_est).fit()

    alpha = model.params["const"]
    beta = model.params["R_market"]
    sigma_eps = model.resid.std(ddof=1)

    event_df = event_df.copy().sort_values("event_day")
    event_df["expected_return"] = alpha + beta * event_df["R_market"]
    event_df["AR"] = event_df["R_stock"] - event_df["expected_return"]
    event_df["CAR"] = event_df["AR"].cumsum()
    event_df["upper_95_AR"] = 1.96 * sigma_eps
    event_df["lower_95_AR"] = -1.96 * sigma_eps

    event_df["ticker"] = ticker
    event_df["name"] = name
    event_df["announcement_date"] = ann_date

    individual_ar_results[ticker] = event_df.copy()

# ---------------------------------------------------------
# 2. Plot individual AR panels with clean 95% thresholds
# ---------------------------------------------------------
tickers = sorted(individual_ar_results.keys())
n = len(tickers)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

for ax, ticker in zip(axes, tickers):
    sub = individual_ar_results[ticker].copy().sort_values("event_day")
    ann_date = pd.Timestamp(sub["announcement_date"].iloc[0]).date()
    band = sub["upper_95_AR"].iloc[0]

    ax.bar(sub["event_day"], sub["AR"], alpha=0.8, width=0.7, label="AR")
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.axhline(band, linestyle=":", linewidth=1)
    ax.axhline(-band, linestyle=":", linewidth=1)

    ax.set_title(f"{ticker}\n{ann_date}")
    ax.grid(alpha=0.3)

for j in range(len(tickers), len(axes)):
    axes[j].axis("off")

fig.suptitle("Individual Event Abnormal Returns with 95% Pointwise Thresholds", y=1.02, fontsize=14)
fig.supxlabel("Trading Days Relative to Announcement")
fig.supylabel("Abnormal Return")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# event_panel must contain: ticker, event_day, AR
aar_panel = event_panel[["ticker", "event_day", "AR"]].copy()

# Average abnormal return by day
aar = aar_panel.groupby("event_day")["AR"].mean().reset_index(name="AAR")
aar = aar.sort_values("event_day")
aar["CAAR"] = aar["AAR"].cumsum()

# ---------------------------------------------------------
# Bootstrap across events
# ---------------------------------------------------------
rng = np.random.default_rng(42)
tickers = aar_panel["ticker"].unique()
event_days = sorted(aar_panel["event_day"].unique())

n_boot = 5000
boot_aar = np.zeros((n_boot, len(event_days)))
boot_caar = np.zeros((n_boot, len(event_days)))

for b in range(n_boot):
    sampled_tickers = rng.choice(tickers, size=len(tickers), replace=True)
    sampled = aar_panel[aar_panel["ticker"].isin(sampled_tickers)].copy()

    # Need resampled event blocks with multiplicity
    blocks = []
    for i, tk in enumerate(sampled_tickers):
        blk = aar_panel[aar_panel["ticker"] == tk].copy()
        blk["boot_id"] = i
        blocks.append(blk)

    sampled_full = pd.concat(blocks, ignore_index=True)

    tmp = sampled_full.groupby("event_day")["AR"].mean().reindex(event_days)
    boot_aar[b, :] = tmp.values
    boot_caar[b, :] = np.cumsum(tmp.values)

aar["AAR_low"] = np.percentile(boot_aar, 2.5, axis=0)
aar["AAR_high"] = np.percentile(boot_aar, 97.5, axis=0)
aar["CAAR_low"] = np.percentile(boot_caar, 2.5, axis=0)
aar["CAAR_high"] = np.percentile(boot_caar, 97.5, axis=0)

# ---------------------------------------------------------
# Plot AAR
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(aar["event_day"], aar["AAR"], marker="o", label="AAR")
plt.fill_between(aar["event_day"], aar["AAR_low"], aar["AAR_high"], alpha=0.2, label="95% bootstrap CI")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.title("Average Abnormal Return Around S&P 500 Inclusion Announcements")
plt.xlabel("Trading Days Relative to Announcement")
plt.ylabel("AAR")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Plot CAAR
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(aar["event_day"], aar["CAAR"], marker="o", label="CAAR")
plt.fill_between(aar["event_day"], aar["CAAR_low"], aar["CAAR_high"], alpha=0.2, label="95% bootstrap CI")
plt.axvline(0, linestyle="--")
plt.axhline(0, linestyle="--")
plt.title("Cumulative Average Abnormal Return Around S&P 500 Inclusion Announcements")
plt.xlabel("Trading Days Relative to Announcement")
plt.ylabel("CAAR")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_1samp

# --------------------------------------------------
# 1. Event-level leakage summaries
# --------------------------------------------------
leakage_rows = []

for ticker, sub in event_panel.groupby("ticker"):
    sub = sub.sort_values("event_day").copy()

    car_pre = sub.loc[sub["event_day"].between(-10, -1), "AR"].sum()
    car_evt = sub.loc[sub["event_day"].between(0, 1), "AR"].sum()
    car_post = sub.loc[sub["event_day"].between(2, 10), "AR"].sum()

    avol_pre = sub.loc[sub["event_day"].between(-10, -1), "AVOL"].mean()
    avol_evt = sub.loc[sub["event_day"].between(0, 1), "AVOL"].mean()
    avol_post = sub.loc[sub["event_day"].between(2, 10), "AVOL"].mean()

    leakage_rows.append({
        "ticker": ticker,
        "announcement_date": sub["announcement_date"].iloc[0],
        "CAR_pre(-10,-1)": car_pre,
        "CAR_evt(0,1)": car_evt,
        "CAR_post(2,10)": car_post,
        "AVOL_pre(-10,-1)": avol_pre,
        "AVOL_evt(0,1)": avol_evt,
        "AVOL_post(2,10)": avol_post
    })

leakage_df = pd.DataFrame(leakage_rows).sort_values("announcement_date")
print(leakage_df.to_string(index=False))

# --------------------------------------------------
# 2. Cross-event t-tests
# --------------------------------------------------
def test_series(x, name):
    x = pd.Series(x).dropna()
    tstat, pval = ttest_1samp(x, 0.0)
    print(f"{name}: mean = {x.mean():.4f}, t = {tstat:.3f}, p = {pval:.4f}, n = {len(x)}")

print("\n===== LEAKAGE TESTS ACROSS EVENTS =====")
test_series(leakage_df["CAR_pre(-10,-1)"], "Pre-event CAR")
test_series(leakage_df["CAR_evt(0,1)"], "Event-window CAR")
test_series(leakage_df["CAR_post(2,10)"], "Post-event CAR")

test_series(leakage_df["AVOL_pre(-10,-1)"], "Pre-event abnormal volume")
test_series(leakage_df["AVOL_evt(0,1)"], "Event-window abnormal volume")
test_series(leakage_df["AVOL_post(2,10)"], "Post-event abnormal volume")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heat_ar = event_panel.pivot(index="ticker", columns="event_day", values="AR")
plt.figure(figsize=(12, 6))
sns.heatmap(heat_ar, center=0, cmap="coolwarm")
plt.title("Abnormal Returns by Stock and Event Day")
plt.xlabel("Event Day")
plt.ylabel("Ticker")
plt.show()

In [ ]:
heat_vol = event_panel.pivot(index="ticker", columns="event_day", values="AVOL")
plt.figure(figsize=(12, 6))
sns.heatmap(heat_vol, center=0, cmap="coolwarm")
plt.title("Abnormal Volume by Stock and Event Day")
plt.xlabel("Event Day")
plt.ylabel("Ticker")
plt.show()

In [ ]:
def compute_event_panel_for_stock(ticker, name, announcement_date, benchmark_ticker, debug=False):
    ann_date = pd.Timestamp(announcement_date)

    start_date = (ann_date - pd.Timedelta(days=DOWNLOAD_BUFFER_BEFORE)).strftime("%Y-%m-%d")
    end_date   = (ann_date + pd.Timedelta(days=DOWNLOAD_BUFFER_AFTER)).strftime("%Y-%m-%d")

    stock = yahoo_download_one(ticker, start_date, end_date)
    market = yahoo_download_one(benchmark_ticker, start_date, end_date)

    if stock is None:
        if debug:
            print(f"{ticker}: stock download returned None")
        return None

    if market is None:
        if debug:
            print(f"{ticker}: market download returned None")
        return None

    df = stock.join(market, how="inner").dropna().copy()

    stock_close_col = f"{ticker}_close"
    stock_vol_col = f"{ticker}_volume"
    mkt_close_col = f"{benchmark_ticker}_close"

    if stock_close_col not in df.columns:
        if debug:
            print(f"{ticker}: missing stock close column")
        return None

    if stock_vol_col not in df.columns:
        if debug:
            print(f"{ticker}: missing stock volume column")
        return None

    if mkt_close_col not in df.columns:
        if debug:
            print(f"{ticker}: missing market close column")
        return None

    df["R_stock"] = df[stock_close_col].pct_change()
    df["R_market"] = df[mkt_close_col].pct_change()
    df["log_volume"] = np.log(df[stock_vol_col].replace(0, np.nan))
    df = df.dropna().copy()

    if len(df) < 170:
        if debug:
            print(f"{ticker}: too few total observations after cleaning ({len(df)})")
        return None

    event_trading_day = nearest_trading_day(df.index, announcement_date)
    event_loc = df.index.get_loc(event_trading_day)
    df["event_day"] = np.arange(len(df)) - event_loc

    est_mask = (df["event_day"] >= EST_START) & (df["event_day"] <= EST_END)
    evt_mask = (df["event_day"] >= EVT_START) & (df["event_day"] <= EVT_END)

    est = df.loc[est_mask].copy()
    evt = df.loc[evt_mask].copy()

    required_evt_len = EVT_END - EVT_START + 1

    if len(est) < 100:
        if debug:
            print(f"{ticker}: too few estimation-window observations ({len(est)})")
        return None

    if len(evt) != required_evt_len:
        if debug:
            print(f"{ticker}: event window incomplete ({len(evt)} vs required {required_evt_len})")
            print(f"  event trading day = {event_trading_day.date()}")
            print(f"  available event-day range = {evt['event_day'].min() if len(evt)>0 else 'NA'} to {evt['event_day'].max() if len(evt)>0 else 'NA'}")
        return None

    X = sm.add_constant(est["R_market"])
    y = est["R_stock"]
    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params["R_market"]
    sigma_eps = model.resid.std(ddof=1)

    evt["expected_return"] = alpha + beta * evt["R_market"]
    evt["AR"] = evt["R_stock"] - evt["expected_return"]

    mean_log_vol_est = est["log_volume"].mean()
    sd_log_vol_est = est["log_volume"].std(ddof=1)

    evt["AVOL"] = evt["log_volume"] - mean_log_vol_est
    evt["SAVOL"] = evt["AVOL"] / sd_log_vol_est if sd_log_vol_est > 0 else np.nan

    evt["ticker"] = ticker
    evt["name"] = name
    evt["announcement_date"] = ann_date
    evt["event_trading_day"] = event_trading_day
    evt["alpha"] = alpha
    evt["beta"] = beta
    evt["sigma_eps"] = sigma_eps
    evt["sigma_logvol"] = sd_log_vol_est

    return evt.reset_index().rename(columns={"index": "date"})

In [ ]:
for e in events:
    _ = compute_event_panel_for_stock(
        ticker=e["ticker"],
        name=e["name"],
        announcement_date=e["announcement_date"],
        benchmark_ticker=benchmark_ticker,
        debug=True
    )

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

# =========================
# 1. SETTINGS
# =========================
stock_ticker = "PLTR"
benchmark_ticker = "^IXIC"   # Nasdaq Composite; use ^GSPC if you prefer S&P 500

start_date = "2023-01-01"
end_date   = "2026-01-31"

# Important Palantir events to annotate
events = {
    "2024-03-04": "Ukraine demining\npartnership",
    "2024-05-30": "DoD CDAO /\nCJADC2 selection",
    "2024-08-08": "Microsoft + Palantir\nclassified AI",
    "2024-09-06": "S&P 500\ninclusion announced",
    "2024-09-20": "Maven military\nservices expansion",
    "2024-11-04": "Q3 earnings +\nraised guidance",
    "2024-12-06": "Booz Allen +\nAnduril defense AI",
    "2024-12-18": "Army Vantage\n$618.9M expansion"
}

# =========================
# 2. DOWNLOAD DATA
# =========================
raw = yf.download(
    [stock_ticker, benchmark_ticker],
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

# yfinance sometimes returns MultiIndex columns
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].copy()
    prices.columns = [stock_ticker]

prices = prices.dropna().copy()
prices.columns = ["PLTR", "Benchmark"]

# =========================
# 3. CORE SERIES
# =========================
df = prices.copy()
df["PLTR_ret"] = df["PLTR"].pct_change()
df["Benchmark_ret"] = df["Benchmark"].pct_change()

# Indexed performance (start = 100)
df["PLTR_index"] = 100 * df["PLTR"] / df["PLTR"].iloc[0]
df["Benchmark_index"] = 100 * df["Benchmark"] / df["Benchmark"].iloc[0]

# Relative strength: PLTR divided by benchmark, rebased to 100
df["Relative_Strength"] = 100 * (df["PLTR"] / df["Benchmark"]) / (df["PLTR"].iloc[0] / df["Benchmark"].iloc[0])

# Rolling metrics
window = 20
df["PLTR_20d_return"] = df["PLTR"].pct_change(window)
df["Benchmark_20d_return"] = df["Benchmark"].pct_change(window)
df["PLTR_20d_vol"] = df["PLTR_ret"].rolling(window).std() * np.sqrt(252)
df["Benchmark_20d_vol"] = df["Benchmark_ret"].rolling(window).std() * np.sqrt(252)

# Rolling beta
rolling_cov = df["PLTR_ret"].rolling(window).cov(df["Benchmark_ret"])
rolling_var = df["Benchmark_ret"].rolling(window).var()
df["Rolling_Beta_20d"] = rolling_cov / rolling_var

# Rolling alpha from simple market model on a moving window
alphas = []
for i in range(len(df)):
    if i < window:
        alphas.append(np.nan)
    else:
        sample = df.iloc[i-window:i].dropna(subset=["PLTR_ret", "Benchmark_ret"])
        X = sm.add_constant(sample["Benchmark_ret"])
        y = sample["PLTR_ret"]
        model = sm.OLS(y, X).fit()
        alphas.append(model.params["const"])
df["Rolling_Alpha_20d"] = alphas

# Linear trend on indexed PLTR price
x = np.arange(len(df))
coef = np.polyfit(x, df["PLTR_index"], 1)
df["PLTR_linear_trend"] = coef[0] * x + coef[1]

# Optional moving averages
df["MA_20"] = df["PLTR"].rolling(20).mean()
df["MA_50"] = df["PLTR"].rolling(50).mean()

# =========================
# 4. HELPER: SNAP EVENT DATES TO TRADING DAYS
# =========================
def nearest_trading_day(index, date_str):
    target = pd.Timestamp(date_str)
    if target in index:
        return target
    # nearest available trading day
    pos = index.get_indexer([target], method="nearest")[0]
    return index[pos]

event_points = []
for d, label in events.items():
    td = nearest_trading_day(df.index, d)
    event_points.append((td, label))

# =========================
# 5. SUMMARY STATS
# =========================
def summarize_trend(data):
    total_return = data["PLTR"].iloc[-1] / data["PLTR"].iloc[0] - 1
    benchmark_return = data["Benchmark"].iloc[-1] / data["Benchmark"].iloc[0] - 1
    excess_return = total_return - benchmark_return

    mean_ret = data["PLTR_ret"].mean() * 252
    vol = data["PLTR_ret"].std() * np.sqrt(252)
    beta_full = data["PLTR_ret"].cov(data["Benchmark_ret"]) / data["Benchmark_ret"].var()

    print("===== TREND SUMMARY =====")
    print(f"Sample: {data.index.min().date()} to {data.index.max().date()}")
    print(f"PLTR total return:       {total_return:.2%}")
    print(f"Benchmark total return:  {benchmark_return:.2%}")
    print(f"PLTR excess return:      {excess_return:.2%}")
    print(f"Annualized mean return:  {mean_ret:.2%}")
    print(f"Annualized volatility:   {vol:.2%}")
    print(f"Full-sample beta:        {beta_full:.3f}")

    latest_rs = data['Relative_Strength'].iloc[-1]
    latest_beta = data["Rolling_Beta_20d"].dropna().iloc[-1] if data["Rolling_Beta_20d"].notna().any() else np.nan
    latest_vol = data["PLTR_20d_vol"].dropna().iloc[-1] if data["PLTR_20d_vol"].notna().any() else np.nan

    print(f"Latest relative strength index: {latest_rs:.2f}")
    print(f"Latest rolling beta (20d):      {latest_beta:.3f}")
    print(f"Latest rolling vol (20d ann.):  {latest_vol:.2%}")

summarize_trend(df.dropna())

# =========================
# 6. MAIN CHART: INDEXED PERFORMANCE + EVENTS
# =========================
plt.figure(figsize=(15, 8))
plt.plot(df.index, df["PLTR_index"], label="PLTR (Indexed to 100)", linewidth=2)
plt.plot(df.index, df["Benchmark_index"], label=f"{benchmark_ticker} (Indexed to 100)", linewidth=1.8)
plt.plot(df.index, df["PLTR_linear_trend"], linestyle="--", label="PLTR linear trend")

for i, (d, label) in enumerate(event_points):
    y = df.loc[d, "PLTR_index"]
    plt.axvline(d, linestyle="--", alpha=0.45)
    offset = 10 if i % 2 == 0 else -15
    plt.annotate(
        label,
        xy=(d, y),
        xytext=(10, offset),
        textcoords="offset points",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", alpha=0.15),
        arrowprops=dict(arrowstyle="->", lw=0.8)
    )

plt.title("Palantir: Full Indexed Price Path with Strategic Event Labels")
plt.xlabel("Date")
plt.ylabel("Indexed Price (Start = 100)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 7. PRICE + MOVING AVERAGES
# =========================
plt.figure(figsize=(15, 7))
plt.plot(df.index, df["PLTR"], label="PLTR Price", linewidth=2)
plt.plot(df.index, df["MA_20"], label="20-day MA", linewidth=1.5)
plt.plot(df.index, df["MA_50"], label="50-day MA", linewidth=1.5)

for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.3)

plt.title("Palantir Price with 20-Day and 50-Day Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 8. RELATIVE STRENGTH
# =========================
plt.figure(figsize=(15, 6))
plt.plot(df.index, df["Relative_Strength"], linewidth=2)
for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.25)
plt.axhline(100, linestyle="--")
plt.title("Palantir Relative Strength vs Benchmark (Rebased to 100)")
plt.xlabel("Date")
plt.ylabel("Relative Strength Index")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 9. ROLLING BETA
# =========================
plt.figure(figsize=(15, 6))
plt.plot(df.index, df["Rolling_Beta_20d"], linewidth=2)
for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.25)
plt.axhline(1.0, linestyle="--")
plt.title("Palantir 20-Day Rolling Beta vs Benchmark")
plt.xlabel("Date")
plt.ylabel("Rolling Beta")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 10. ROLLING VOLATILITY
# =========================
plt.figure(figsize=(15, 6))
plt.plot(df.index, df["PLTR_20d_vol"], label="PLTR 20d annualized vol", linewidth=2)
plt.plot(df.index, df["Benchmark_20d_vol"], label="Benchmark 20d annualized vol", linewidth=1.8)
for d, label in event_points:
    plt.axvline(d, linestyle="--", alpha=0.25)
plt.title("Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 11. 20-DAY FORWARD RETURN AFTER EACH EVENT
# =========================
print("\n===== EVENT-BY-EVENT FORWARD RETURN CHECK =====")
rows = []
for d, label in event_points:
    if d not in df.index:
        continue
    start_px = df.loc[d, "PLTR"]
    # nearest 20 trading days later
    idx_pos = df.index.get_loc(d)
    end_pos = min(idx_pos + 20, len(df) - 1)
    end_d = df.index[end_pos]
    end_px = df.iloc[end_pos]["PLTR"]
    fwd_ret = end_px / start_px - 1

    rows.append({
        "Event_Date": d.date(),
        "Label": label.replace("\n", " "),
        "PLTR_Price_On_Event": round(start_px, 2),
        "20D_Later": end_d.date(),
        "20D_Forward_Return": fwd_ret
    })

event_forward = pd.DataFrame(rows)
print(event_forward.to_string(index=False))

# =========================
# 12. OPTIONAL: SIMPLE TREND REGIMES
# =========================
df["Above_MA20"] = (df["PLTR"] > df["MA_20"]).astype(int)
df["Above_MA50"] = (df["PLTR"] > df["MA_50"]).astype(int)

print("\n===== LATEST TREND REGIME =====")
latest = df.dropna().iloc[-1]
print(f"Latest date:            {df.dropna().index[-1].date()}")
print(f"Price above 20d MA:     {bool(latest['Above_MA20'])}")
print(f"Price above 50d MA:     {bool(latest['Above_MA50'])}")
print(f"20d rolling beta:       {latest['Rolling_Beta_20d']:.3f}")
print(f"20d rolling vol:        {latest['PLTR_20d_vol']:.2%}")
print(f"Relative strength idx:  {latest['Relative_Strength']:.2f}")